# Multi-hop questions with `QueryPlanEngine`

`QueryPlanEngine` wraps any other engine. Given a question that cannot be answered
from one retrieval, it:

1. decomposes the question into a DAG of subqueries,
2. answers each subquery through the wrapped engine, in dependency order,
3. rewrites dependent subqueries so they are self-contained, injecting the answers
   they depend on,
4. returns the answer of the plan's final (sink) subquery.

The wrapped engine never learns it is being planned for; it just receives ordinary
queries. So the planner composes with local, global, naive or mix search
interchangeably.

**Environment:** `OPENAI_API_KEY`, `LLM_MODEL_NAME`, `EMBEDDER_MODEL_NAME`, and
optionally `OPENAI_BASE_URL`.

In [7]:
import os
from pathlib import Path

from ragu import (
    ArtifactsExtractorLLM,
    BuilderArguments,
    KnowledgeGraph,
    LocalSearchEngine,
    QueryPlanEngine,
    Settings,
    SimpleChunker,
)
from ragu.models.embedder import EmbedderOpenAI
from ragu.models.llm import LLMOpenAI
from ragu.models.openai import CachedAsyncOpenAI
from ragu.search_engine.local_search import LocalParams
from ragu.utils.ragu_utils import read_text_from_files

DATA_DIR = Path("data/en")

# Two-hop questions
QUESTIONS = [
    "Who created the C programming language, and where did that person's father work?",
    "What did the person who died on October 12, 2011 create, and who did they work with?",
]

In [ ]:
Settings.language = "english"
Settings.storage_folder = "ragu_working_dir/query_plan_example"


client = CachedAsyncOpenAI(
    base_url=os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1"),
    api_key=os.environ["OPENAI_API_KEY"],
    rate_max_simultaneous=10,
    rate_max_per_minute=100,
)
llm = LLMOpenAI(client=client, model_name=os.environ["LLM_MODEL_NAME"])
embedder = EmbedderOpenAI(client=client, model_name=os.environ["EMBEDDER_MODEL_NAME"])
await embedder.initialize()

## Build the graph

The expensive cell. Run once.

In [ ]:
knowledge_graph = KnowledgeGraph(
    llm=llm,
    embedder=embedder,
    chunker=SimpleChunker(max_chunk_size=1000),
    artifact_extractor=ArtifactsExtractorLLM(llm=llm, embedder=embedder),
    builder_settings=BuilderArguments(),
)
await knowledge_graph.build_from_docs(read_text_from_files(DATA_DIR))

In [10]:
base_engine = LocalSearchEngine(
    llm=llm, 
    knowledge_graph=knowledge_graph, 
    embedder=embedder
)
planner = QueryPlanEngine(base_engine)

# Parameters are forwarded verbatim to the wrapped engine for every subquery.
params = LocalParams(top_k=10)

## Inspect the plan

`process_query` decomposes without executing, which is how you debug a planner
that produces the wrong hops.

In [11]:
for subquery in await planner.process_query(QUESTIONS[0]):
    depends = ", ".join(subquery.depends_on) or "-"
    print(f"{subquery.id} intent={subquery.intent} depends_on={depends}")
    print(f"    {subquery.query}")

QueryPlan decompose: 100%|██████████| 1/1 [00:04<00:00,  4.92s/it]

q1 intent=lookup depends_on=-
    Who created the C programming language?
q2 intent=lookup depends_on=q1
    Who was the father of the person identified as the creator of the C programming language?
q3 intent=lookup depends_on=q2
    Where did that father work?


In [15]:
COMPLEX_QUESTION = """
Identify a European scientist who was born in a city located on a river that flows into a sea whose coastline includes the capital of a 
country that joined the European Union in the same year the scientist died. 
Then identify the scientist's most significant discovery, find the year in which they published their first paper on that discovery,
and calculate how many years elapsed between that publication and the country's accession to the European Union. 
"""

for subquery in await planner.process_query([COMPLEX_QUESTION]):
    depends = ", ".join(subquery.depends_on) or "-"
    print(f"{subquery.id} intent={subquery.intent} depends_on={depends}")
    print(f"    {subquery.query}")

QueryPlan decompose: 100%|██████████| 1/1 [00:14<00:00, 14.62s/it]

q1 intent=lookup depends_on=-
    Retrieve the countries that have joined the European Union and the year in which each country acceded.
q2 intent=lookup depends_on=q1
    Identify the capital city of each country retrieved in q1.
q3 intent=filtering depends_on=q2
    Determine which capitals from q2 lie on a sea coastline and identify the corresponding sea for each coastal capital.
q4 intent=filtering depends_on=q1
    Identify European scientists whose year of death matches an EU accession year retrieved in q1.
q5 intent=lookup depends_on=q4
    Identify the birth city of each candidate scientist from q4.
q6 intent=lookup depends_on=q5
    Identify the river on which each candidate scientist's birth city is located.
q7 intent=lookup depends_on=q6
    Identify the sea into which each river from q6 flows.
q8 intent=reasoning depends_on=q1, q3, q4, q7
    Select the scientist and country pair for which the scientist's death year equals the country's EU accession year and the river throu

## Answer with the planner

Batched: subqueries from different top-level questions that become ready at the
same step are answered in one call.

In [ ]:
for response in await planner.batch_query(QUESTIONS, params):
    print(f"Q: {response.query}")
    print(f"A: {response.response}\n")

In [ ]:
await knowledge_graph.index.close()